# Tarefa 2 - IMDb Top 250

https://www.imdb.com/chart/top/

In [6]:
!pip install selenium webdriver-manager


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time, json

In [8]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
print("driver iniciado")

driver iniciado


In [9]:
driver.get("https://www.imdb.com/chart/top/") #URL do Top 250
time.sleep(3)

soup = BeautifulSoup(driver.page_source, "html.parser")

top250 = [] #coleta dos dados
filmes_html = soup.select("li.ipc-metadata-list-summary-item")


for i, filme in enumerate(filmes_html, start=1):
    tag_titulo = filme.find("h3")
    tag_link   = filme.find("a", href=True)

    titulo = tag_titulo.get_text(strip=True) if tag_titulo else "N/A"
    url    = "https://www.imdb.com" + tag_link["href"] if tag_link else "N/A"

    if ". " in titulo: 
        titulo = titulo.split(". ", 1)[1]

    top250.append({"ranking": i, "titulo": titulo, "url": url})

print(f"Filmes encontrados: {len(top250)}")
for f in top250[:5]:
    print(f"#{f['ranking']} {f['titulo']}")

Filmes encontrados: 0


In [13]:
#scraping das pgs individuais de cada filme
filmes_detalhados = []

for i, filme in enumerate(top250, start=1):
    resultado = {
        "ranking"        : filme["ranking"],
        "titulo"         : filme["titulo"],
        "ano_lancamento" : None,
        "poster_url"     : None,
        "nota_imdb"      : None,
        "generos"        : [],
        "diretores"      : [],
        "url"            : filme["url"]
    }

    try:
        driver.get(filme["url"])
        time.sleep(2)

        s = BeautifulSoup(driver.page_source, "html.parser")

        script_ld = s.find("script", type="application/ld+json")
        if script_ld:
            ld = json.loads(script_ld.string)

            resultado["titulo"] = ld.get("name", filme["titulo"])

            data = ld.get("datePublished", "")
            if data:
                resultado["ano_lancamento"] = int(data[:4])

            imagem = ld.get("image")
            resultado["poster_url"] = imagem.get("url") if isinstance(imagem, dict) else imagem

            resultado["nota_imdb"] = ld.get("aggregateRating", {}).get("ratingValue")

            generos = ld.get("genre", [])
            resultado["generos"] = [generos] if isinstance(generos, str) else generos

            diretores = ld.get("director", [])
            if isinstance(diretores, dict):
                diretores = [diretores]
            resultado["diretores"] = [d.get("name") for d in diretores if d.get("name")]

    except Exception as e:
        print(f"Erro em {filme['titulo']}: {e}")

    filmes_detalhados.append(resultado)

    if i % 25 == 0:
        print(f"{i}/250 filmes coletados")

driver.quit()
print(f"Concluído. {len(filmes_detalhados)} filmes coletados.")
            

Concluído. 0 filmes coletados.
